# Module 10: Reading ACF and PACF as Pictures

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 16](../../Beginner/Topic_16_Autocorrelation.md) showed that a
series resembles its own recent past. The autocorrelation function puts that on
a chart, one bar per lag.

At this level it earns its keep in two concrete ways. It tells you whether your
model has finished, and it tells you whether your confidence intervals can be
believed.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

g = final[final["agency_id"] == "A012"].copy()
g["rate"] = 100 * g["n_uof"] / g["n_arrests"]
g["t"] = np.arange(len(g))
g["mon"] = g["year_month"].str[5:7].astype(int)
g = g[g["year_month"] <= "2025-12"]        # whole years
print(f"{len(g)} months")

## 2. The ACF of a raw series is mostly calendar

Each bar is the correlation between the series and itself shifted by that many
months. Anything inside the shaded band is indistinguishable from zero.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

band = 1.96 / np.sqrt(len(g))
a = acf(grandview.dropna(), nlags=24, fft=False)

print(f"noise band: plus or minus {band:.2f}\n")
for k in [1, 3, 6, 9, 12, 18, 24]:
    mark = "  <-- outside" if abs(a[k]) > band else ""
    print(f"  lag {k:2d}: {a[k]:+.2f}{mark}")

A big positive value at lag 1, a big **negative** one at lag 6, and a big
positive one again at lag 12. That is a wave with a twelve month period, and it
is the season, not memory in any interesting sense.

An ACF computed on a raw seasonal series will always look like this and will
always tell you the same thing you already knew.

## 3. The useful version: run it on the residuals

The question worth asking is not "does the series have structure" but **"does
my model still have structure left in it"**. So fit the model and run the ACF
on what it could not explain.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_ljungbox

for name, formula in [("trend only", "np.log(rate) ~ t"),
                      ("trend and month terms", "np.log(rate) ~ t + C(mon)")]:
    res = smf.ols(formula, data=g).fit().resid
    r = acf(res, nlags=24, fft=False)
    outside = int((np.abs(r[1:]) > band).sum())
    p = acorr_ljungbox(res, lags=[12], return_df=True)["lb_pvalue"].iloc[0]
    print(f"{name:24s} lags outside the band: {outside:2d} of 24   "
          f"Ljung Box p at lag 12: {p:.4f}")

**Ljung Box** tests all the lags at once, so you do not have to eyeball
twenty four bars. A small p value says there is structure left.

With a trend only, the residuals still contain the whole seasonal cycle and the
test rejects overwhelmingly. Add month terms and the test no longer rejects.
That is the model telling you it is finished, and it is a far more useful
statement than any coefficient in it.

## 4. PACF, and what it is for

The ACF at lag 3 includes everything that reached lag 3 through lags 1 and 2.
The **partial** autocorrelation strips that out and reports only what lag 3 adds
on its own.

The practical use is choosing how many lags a model needs, which matters in the
Advanced series. At this level, one glance is enough: if the PACF has a single
large value at lag 1 and little after it, the series is carrying simple
persistence rather than a long memory.

In [ ]:
p = pacf(grandview.dropna(), nlags=12)
print("PACF of the raw counts")
for k in [1, 2, 3, 4, 12]:
    mark = "  <-- outside" if abs(p[k]) > band else ""
    print(f"  lag {k:2d}: {p[k]:+.2f}{mark}")

## 5. Why any of this affects your conclusions

Ordinary regression assumes the errors are independent. If they are not, the
standard errors are wrong, and therefore so is every confidence interval and
every p value.

Check by comparing ordinary standard errors against ones that allow for
autocorrelation.

In [ ]:
per_year = lambda b: 100 * (np.exp(12 * b) - 1)

for name, formula in [("trend only", "np.log(rate) ~ t"),
                      ("trend and month terms", "np.log(rate) ~ t + C(mon)")]:
    plain = smf.ols(formula, data=g).fit()
    robust = smf.ols(formula, data=g).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    print(f"\n{name}")
    for lab, fit in [("ordinary", plain), ("autocorrelation robust", robust)]:
        lo, hi = fit.conf_int().loc["t"]
        print(f"  {lab:24s} {per_year(fit.params['t']):+.2f} percent a year  "
              f"[{per_year(lo):+.2f}, {per_year(hi):+.2f}]  "
              f"width {per_year(hi) - per_year(lo):.2f}")

Two things to take from this, and the second one contradicts a common piece of
advice.

**First, the estimate does not move.** Autocorrelation does not bias the slope.
It affects only how certain you are allowed to be about it.

**Second, the interval here gets narrower, not wider.** The usual warning is
that autocorrelation makes ordinary standard errors too small. That warning
assumes **positive** autocorrelation. These residuals are negatively correlated
at most lags, and negative autocorrelation makes ordinary standard errors too
**large**.

So the rule is not "autocorrelation inflates your confidence". It is
**"ignoring autocorrelation gives you the wrong interval, and you have to look
to find out in which direction"**.

## 6. The order of operations

1. Fit the model you think is right.
2. Run the ACF and Ljung Box **on the residuals**.
3. If there is structure left, **fix the model** rather than patching the
   standard errors. Missing seasonal terms is a modelling error, not a
   standard error problem.
4. Once the residuals are clean, check whether robust standard errors change
   anything. If they do, report those.

## Exercise

Run the same check on Tarnbridge. Its series contains one extraordinary month.
Does the Ljung Box test notice, and does removing that month change the verdict?

In [ ]:
# Fill in the blanks, then run.
AGENCY = None              # try "A002"
DROP_THE_EVENT = None      # try False, then True

if AGENCY is not None and DROP_THE_EVENT is not None:
    d = final[final["agency_id"] == AGENCY].copy()
    d["rate"] = 100 * d["n_uof"] / d["n_arrests"]
    d["mon"] = d["year_month"].str[5:7].astype(int)
    d = d[d["year_month"] <= "2025-12"]
    if DROP_THE_EVENT:
        d = d[d["year_month"] != "2021-06"]
    d["t"] = np.arange(len(d))

    res = smf.ols("np.log(rate) ~ t + C(mon)", data=d).fit().resid
    p = acorr_ljungbox(res, lags=[12], return_df=True)["lb_pvalue"].iloc[0]
    print(f"months: {len(d)}   Ljung Box p at lag 12: {p:.4f}")
    print(f"largest residual: {res.abs().max():.2f}")
else:
    print("Set AGENCY and DROP_THE_EVENT above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
DROP_THE_EVENT = False      # then True
```

The Ljung Box test rejects **both** times, at p = 0.030 with the event left in
and p = 0.023 with it taken out. Removing the outlier does not fix it, and
barely moves the p value at all.

Meanwhile the largest residual falls from 1.29 to 0.59. So the event dominates
the **size** of the residuals and contributes essentially nothing to their
**correlation across time**.

Two separate findings follow, and neither would have been visible from the
other test.

First, the two checks look for different things. An outlier is one large value
sitting on its own; autocorrelation is a pattern linking neighbouring values.
Passing or failing one says nothing about the other, so run both. Beginner
[Topic 11](../../Beginner/Topic_11_Outliers_And_Spikes.md) and
[Module 8](../Module_08_Rolling_Statistics_And_Control_Limits.md) cover the
outlier side.

Second, Tarnbridge genuinely has serial structure that a trend and month terms
do not capture, and the honest response is to keep modelling rather than to
declare the model finished. Fitting that structure is what the Advanced series
does with ARIMA and regression with correlated errors.

</details>

---

**Next:** [Module 11, Lead and Lag Between Two Series](Module_11_Lead_And_Lag.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*